In [ ]:
# Block 0 — setup. Code from GitHub onto the VM disk; data + embeddings on Drive.
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

# numpy/scipy/torch are filtered out: Colab's torch is built against its own numpy
!grep -viE '^(torch|torchvision|torchaudio|nvidia-|numpy|scipy|#)' requirements.txt > req_colab.txt
!pip install -q -r req_colab.txt

In [ ]:
# Block 1 — the WHOLE pan-cancer manifest (every peptide of every arm) and its embedding cache key.
# Every arm frame (colorectal / lung / melanoma / pooled, any fold) is a subset of the manifest, so the embeddings are computed
# ONCE here and every arm reads this cache. The key is a hash of the manifest's (uid, sequence) pairs.
import torch
from src import config, data, embed, pancancer_split as ps

NAME = "pancancer_cedar_v1"
manifest = ps.load_manifest(config.DATA_DIR, NAME)
EID = ps.emb_id(manifest)
print("DATA_DIR :", config.DATA_DIR)
print("cache key:", EID)
print("sequences:", len(manifest), "| residues:", int(manifest["length"].sum()))
print("GPU      :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# Block 2 — SMOKE TEST: tiny 8M checkpoint, a few seconds. Run once before committing to the 650M models.
embed.build_cache("esm2_8m", manifest, build_id=EID)
c = embed.EmbCache("esm2_8m", EID, staged=False)
x, lens = c.stack(manifest["uid"].head(4), int(manifest["length"].max()))
print("stack:", x.shape, x.dtype, "lengths:", lens, "| dim:", c.dim)

In [ ]:
# Block 3 — ProtTrans (ProtT5-XL, encoder-only fp16, 1024-d). The paper's primary feature set.
embed.build_cache("prott5", manifest, build_id=EID)

In [ ]:
# Block 4 — ESM-2 650M (1280-d)
embed.build_cache("esm2_650m", manifest, build_id=EID)

In [ ]:
# Block 5 — ESM-1b (1280-d)
embed.build_cache("esm1b", manifest, build_id=EID)

In [ ]:
# Block 6 — what is cached
for plm in ["prott5", "esm2_650m", "esm1b", "esm2_8m"]:
    ok = embed.is_cached(plm, EID)
    print(f"{plm:12s} {'cached' if ok else 'MISSING'}")